# Compute FLOPs Analysis

In [1]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from fvcore.nn import FlopCountAnalysis

class SimpleMLP(pl.LightningModule):
    def __init__(self, in_features, out_features, bias):
        super().__init__()
        self.save_hyperparameters()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
    def forward(self, x):
        return self.linear(x)

class LowRankLinear(pl.LightningModule):
    def __init__(self, in_features, out_features, rank, bias):
        super().__init__()
        self.save_hyperparameters()
        self.linear_A = nn.Linear(in_features, rank, bias=False)
        self.linear_B = nn.Linear(rank, out_features, bias=bias)
    def forward(self, x):
        return self.linear_B(self.linear_A(x))

def analyze_layer(in_features, out_features, ranks, bias=True):
    n_params = lambda m: sum(p.numel() for p in m.parameters())
    get_flops = lambda m, x: FlopCountAnalysis(m, (x,)).total()

    full = SimpleMLP(in_features, out_features, bias)
    x = torch.randn(1, in_features)
    full_params = n_params(full)
    full_flops = get_flops(full, x)

    print(f"\n{'='*75}")
    print(f"Layer: {in_features} -> {out_features}  (bias={bias})")
    print(f"Full-rank: {full_params:,} params | {full_flops/1e6:.2f} MFLOPs")
    print(f"{'='*75}")
    print(f"{'Rank':>6} | {'Params':>10} | {'Param %':>9} | {'MFLOPs':>10} | {'FLOP %':>9}")
    print(f"{'-'*6}-+-{'-'*10}-+-{'-'*9}-+-{'-'*10}-+-{'-'*9}")

    for r in ranks:
        low = LowRankLinear(in_features, out_features, r, bias=bias)
        lp = n_params(low)
        lf = get_flops(low, x)
        print(f"{r:>6} | {lp:>10,} | {100*lp/full_params:>8.2f}% | {lf/1e6:>10.2f} | {100*lf/full_flops:>8.2f}%")

RANKS = [2**i for i in range(1, 11)]

analyze_layer(1024, 1024, RANKS)
analyze_layer(1024, 10,   RANKS + [10])


Layer: 1024 -> 1024  (bias=True)
Full-rank: 1,049,600 params | 1.05 MFLOPs
  Rank |     Params |   Param % |     MFLOPs |    FLOP %
-------+------------+-----------+------------+----------
     2 |      5,120 |     0.49% |       0.00 |     0.39%
     4 |      9,216 |     0.88% |       0.01 |     0.78%
     8 |     17,408 |     1.66% |       0.02 |     1.56%
    16 |     33,792 |     3.22% |       0.03 |     3.12%
    32 |     66,560 |     6.34% |       0.07 |     6.25%
    64 |    132,096 |    12.59% |       0.13 |    12.50%
   128 |    263,168 |    25.07% |       0.26 |    25.00%
   256 |    525,312 |    50.05% |       0.52 |    50.00%
   512 |  1,049,600 |   100.00% |       1.05 |   100.00%
  1024 |  2,098,176 |   199.90% |       2.10 |   200.00%

Layer: 1024 -> 10  (bias=True)
Full-rank: 10,250 params | 0.01 MFLOPs
  Rank |     Params |   Param % |     MFLOPs |    FLOP %
-------+------------+-----------+------------+----------
     2 |      2,078 |    20.27% |       0.00 |    20.20